# 零知识证明实践：证明 Alice 知道 `x^3 + x + 5 = out` 的解

本实验参考教材第四章中关于零知识证明、Schnorr 协议和 Fiat-Shamir 变换的内容，实现一个不依赖第三方库的教学版非交互式零知识证明。

实验命题：公开值 `out = 35`，Alice 希望证明自己知道一个秘密值 `x`，满足：

`x^3 + x + 5 = out`

本实验中 Alice 的见证为 `x = 3`，因为 `3^3 + 3 + 5 = 35`。

## 1. 实验思路

教材中介绍的 Schnorr 协议可以证明“我知道某个离散对数 `x`”，而不直接公开 `x`。为了证明三次方程，我们把等式拆成椭圆曲线群上的关系：

- 令 `G` 是公开的椭圆曲线基点。
- Alice 计算 `A = xG`，`B = xA = x^2G`，`C = xB = x^3G`。
- 验证者检查 `C + A + 5G == outG`，这对应 `x^3G + xG + 5G == outG`。
- 还需要证明 `A, B, C` 确实由同一个秘密 `x` 生成，因此使用两个非交互式 Chaum-Pedersen 证明：
  - 证明 `log_G(A) = log_A(B)`，即 `A = xG` 且 `B = xA` 使用同一个 `x`。
  - 证明 `log_G(A) = log_B(C)`，即 `A = xG` 且 `C = xB` 使用同一个 `x`。

Chaum-Pedersen 证明可以看作 Schnorr 协议的扩展版，用来证明两个离散对数相等。本实验使用 Fiat-Shamir 变换，把交互式挑战 `c` 改成哈希函数输出，从而得到非交互式证明。

注意：这是教学版实现，目的是展示证明生成和验证流程；生产环境应使用成熟 zkSNARK、Bulletproofs 或其他经过审计的密码库。由于本题 `x = 3` 很小，且方程也很简单，现实中验证者可以直接猜出 `x`，所以本实验重点是协议机制，而不是隐藏这个特定小数值。

In [2]:
import hashlib
import secrets
from pprint import pprint

# secp256k1 椭圆曲线参数。
# 曲线方程为：y^2 = x^3 + 7 (mod P)。
# 这里直接使用公开标准参数，避免依赖第三方椭圆曲线库。
P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
A_CURVE = 0
B_CURVE = 7

# secp256k1 的公开基点 G。
G = (
    0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
    0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8,
)

# 用 None 表示无穷远点，也就是椭圆曲线群的单位元。
INF = None

上面的代码定义了椭圆曲线的公开参数。实验中所有人都知道这些参数，Alice 的秘密只有 `x`。椭圆曲线离散对数问题保证：已知 `G` 和 `xG`，一般情况下无法有效反推出 `x`。

In [3]:
def mod_inv(value, modulus):
    """计算 value 在模 modulus 下的乘法逆元。"""
    return pow(value % modulus, -1, modulus)


def is_on_curve(point):
    """检查点是否在 secp256k1 曲线上。"""
    if point is INF:
        return True
    x, y = point
    return (y * y - (x * x * x + A_CURVE * x + B_CURVE)) % P == 0


def point_neg(point):
    """计算椭圆曲线点的相反数。"""
    if point is INF:
        return INF
    x, y = point
    return (x, (-y) % P)


def point_add(left, right):
    """椭圆曲线点加法。"""
    if left is INF:
        return right
    if right is INF:
        return left

    x1, y1 = left
    x2, y2 = right

    # 如果两个点互为相反数，相加得到无穷远点。
    if x1 == x2 and (y1 + y2) % P == 0:
        return INF

    if left == right:
        # 点加倍时的斜率。
        slope = (3 * x1 * x1 + A_CURVE) * mod_inv(2 * y1, P) % P
    else:
        # 普通点加法时的斜率。
        slope = (y2 - y1) * mod_inv(x2 - x1, P) % P

    x3 = (slope * slope - x1 - x2) % P
    y3 = (slope * (x1 - x3) - y1) % P
    return (x3, y3)


def scalar_mul(k, point):
    """椭圆曲线标量乘法：计算 k * point。"""
    if point is INF or k % N == 0:
        return INF
    if k < 0:
        return scalar_mul(-k, point_neg(point))

    k = k % N
    result = INF
    addend = point

    # 二进制 double-and-add 算法。
    while k:
        if k & 1:
            result = point_add(result, addend)
        addend = point_add(addend, addend)
        k >>= 1
    return result


# 基本自检：基点应在曲线上，并且 N * G 是无穷远点。
assert is_on_curve(G)
assert scalar_mul(N, G) is INF
print("椭圆曲线基础运算自检通过。")

椭圆曲线基础运算自检通过。


这一段实现椭圆曲线点加法和标量乘法。教材中用 `aG` 表示把秘密数 `a` 映射成椭圆曲线点，本实验中的 `scalar_mul(a, G)` 就是在计算这个值。

In [4]:
def int_to_32_bytes(value):
    """把整数编码成 32 字节，用于哈希输入。"""
    return int(value % N).to_bytes(32, "big")


def encode_point(point):
    """把椭圆曲线点编码成字节串。"""
    if point is INF:
        return b"\x00"
    x, y = point
    return b"\x04" + x.to_bytes(32, "big") + y.to_bytes(32, "big")


def encode_item(item):
    """给哈希函数准备无歧义编码，防止不同类型数据拼接后产生混淆。"""
    if item is INF or isinstance(item, tuple):
        data = b"P" + encode_point(item)
    elif isinstance(item, int):
        data = b"I" + int_to_32_bytes(item)
    elif isinstance(item, str):
        data = b"S" + item.encode("utf-8")
    else:
        raise TypeError(f"不支持的哈希输入类型：{type(item)}")
    return len(data).to_bytes(4, "big") + data


def hash_challenge(label, *items):
    """Fiat-Shamir 变换：用 SHA-256 生成不可预测挑战 c。"""
    digest = hashlib.sha256()
    digest.update(encode_item(label))
    for item in items:
        digest.update(encode_item(item))
    return int.from_bytes(digest.digest(), "big") % N


def point_to_hex(point):
    """把点转成便于查看的十六进制形式。"""
    if point is INF:
        return "INF"
    x, y = point
    return {"x": hex(x), "y": hex(y)}

Fiat-Shamir 变换的核心是：证明者不再等待验证者发送随机挑战，而是把公开语句、承诺点等内容输入哈希函数，得到挑战 `c`。只要哈希函数抗预映像、输出不可预测，证明者就难以提前操控挑战。

In [5]:
def prove_equal_logs(base1, result1, base2, result2, witness, label):
    """
    生成非交互式 Chaum-Pedersen 证明。

    证明目标：存在同一个 witness，使得：
        result1 = witness * base1
        result2 = witness * base2

    这等价于证明两个离散对数相等，但不公开 witness。
    """
    # 1. 随机选择盲因子 r，并计算两个承诺点。
    r = secrets.randbelow(N - 1) + 1
    t1 = scalar_mul(r, base1)
    t2 = scalar_mul(r, base2)

    # 2. 用哈希生成挑战 c。
    c = hash_challenge(label, base1, result1, base2, result2, t1, t2)

    # 3. 计算响应 z = r + c * witness。
    z = (r + c * witness) % N
    return {"t1": t1, "t2": t2, "z": z}


def verify_equal_logs(base1, result1, base2, result2, proof, label):
    """验证非交互式 Chaum-Pedersen 证明。"""
    t1 = proof["t1"]
    t2 = proof["t2"]
    z = proof["z"]

    # 验证所有点都在曲线上，避免无效点攻击。
    points = [base1, result1, base2, result2, t1, t2]
    if not all(is_on_curve(point) for point in points):
        return False

    # 验证者用同样的公开输入重新计算挑战 c。
    c = hash_challenge(label, base1, result1, base2, result2, t1, t2)

    # 检查 Schnorr 型等式：z * base = t + c * result。
    check1 = scalar_mul(z, base1) == point_add(t1, scalar_mul(c, result1))
    check2 = scalar_mul(z, base2) == point_add(t2, scalar_mul(c, result2))
    return check1 and check2

如果证明者真的知道 `witness`，则有：

`z * base1 = (r + c*witness)base1 = r*base1 + c*(witness*base1) = t1 + c*result1`

第二个等式同理。验证者只看到 `t1, t2, z`，不能直接从中恢复 `witness`。

In [6]:
def generate_polynomial_proof(x, out):
    """
    Alice 生成证明。

    私密见证：x
    公开声明：out
    目标：证明 x^3 + x + 5 = out，但不直接公开 x。
    """
    # 实验中先在整数范围检查见证是否满足题目方程。
    if x ** 3 + x + 5 != out:
        raise ValueError("见证 x 不满足 x^3 + x + 5 = out，不能生成诚实证明。")

    # 把 x、x^2、x^3 映射到椭圆曲线点。
    A_point = scalar_mul(x, G)          # A = xG
    B_point = scalar_mul(x, A_point)    # B = xA = x^2G
    C_point = scalar_mul(x, B_point)    # C = xB = x^3G

    # 证明 A 和 B 使用同一个 x：log_G(A) = log_A(B)。
    cp1 = prove_equal_logs(
        G,
        A_point,
        A_point,
        B_point,
        x,
        label=f"cp1|out={out}",
    )

    # 证明 A 和 C 使用同一个 x：log_G(A) = log_B(C)。
    cp2 = prove_equal_logs(
        G,
        A_point,
        B_point,
        C_point,
        x,
        label=f"cp2|out={out}",
    )

    return {
        "out": out,
        "A": A_point,
        "B": B_point,
        "C": C_point,
        "cp1": cp1,
        "cp2": cp2,
    }


def verify_polynomial_proof(proof):
    """验证者验证 Alice 发来的证明。验证过程中不需要知道 x。"""
    out = proof["out"]
    A_point = proof["A"]
    B_point = proof["B"]
    C_point = proof["C"]

    # 1. 检查公开证明中的点是否合法。
    if not all(is_on_curve(point) for point in [A_point, B_point, C_point]):
        return False
    if A_point is INF or B_point is INF or C_point is INF:
        return False

    # 2. 检查两个 Chaum-Pedersen 证明，确认 A、B、C 由同一个 x 关联生成。
    cp1_ok = verify_equal_logs(
        G,
        A_point,
        A_point,
        B_point,
        proof["cp1"],
        label=f"cp1|out={out}",
    )

    cp2_ok = verify_equal_logs(
        G,
        A_point,
        B_point,
        C_point,
        proof["cp2"],
        label=f"cp2|out={out}",
    )

    # 3. 检查多项式输出：C + A + 5G 是否等于 outG。
    left = point_add(point_add(C_point, A_point), scalar_mul(5, G))
    right = scalar_mul(out, G)
    equation_ok = left == right

    return cp1_ok and cp2_ok and equation_ok

证明生成函数只由 Alice 调用，因此它接收秘密 `x`。验证函数只接收证明对象，里面没有明文 `x` 字段。验证者依靠椭圆曲线点关系和两个零知识子证明判断 Alice 是否知道满足方程的 `x`。

In [7]:
def printable_proof(proof):
    """把证明转成便于阅读的形式。实际协议中会把这些字段序列化发送给验证者。"""
    return {
        "out": proof["out"],
        "A": point_to_hex(proof["A"]),
        "B": point_to_hex(proof["B"]),
        "C": point_to_hex(proof["C"]),
        "cp1": {
            "t1": point_to_hex(proof["cp1"]["t1"]),
            "t2": point_to_hex(proof["cp1"]["t2"]),
            "z": hex(proof["cp1"]["z"]),
        },
        "cp2": {
            "t1": point_to_hex(proof["cp2"]["t1"]),
            "t2": point_to_hex(proof["cp2"]["t2"]),
            "z": hex(proof["cp2"]["z"]),
        },
    }


# Alice 的秘密见证和公开输出。
x = 3
out = 35

# Alice 生成证明，并发送给验证者。
proof = generate_polynomial_proof(x, out)

print("生成的证明中没有直接出现 x =", x)
print("证明字段如下：")
pprint(printable_proof(proof), sort_dicts=False)

# 验证者验证证明。
result = verify_polynomial_proof(proof)
print("\n验证结果：", result)

生成的证明中没有直接出现 x = 3
证明字段如下：
{'out': 35,
 'A': {'x': '0xf9308a019258c31049344f85f89d5229b531c845836f99b08601f113bce036f9',
       'y': '0x388f7b0f632de8140fe337e62a37f3566500a99934c2231b6cb9fd7584b8e672'},
 'B': {'x': '0xacd484e2f0c7f65309ad178a9f559abde09796974c57e714c35f110dfc27ccbe',
       'y': '0xcc338921b0a7d9fd64380971763b61e9add888a4375f8e0f05cc262ac64f9c37'},
 'C': {'x': '0xdaed4f2be3a8bf278e70132fb0beb7522f570e144bf615c07e996d443dee8729',
       'y': '0xa69dce4a7d6c98e8d4a1aca87ef8d7003f83c230f3afa726ab40e52290be1c55'},
 'cp1': {'t1': {'x': '0x692203d2a4b50173513268dcf4886d3338dc90938c04ac34453779aa38bde425',
                'y': '0x2656d4ae2f420dd8072d78c7e872b1cf261a774bfb66fd89f9aa1d2c645c26b1'},
         't2': {'x': '0xbe2348fedf6497bbdd1433b5459259f973a495d18b2ea1f5c43d47c8c0c593cd',
                'y': '0x87a76e2a266fae00aea9c6f675bbcdfc236bc6c4425d9499d4b330ddb17fcf42'},
         'z': '0xda6808347ede395d3e06039646940d77b1ba92c1f6519e00a1e2f2228e528de0'},
 'cp2': {'t1': 

如果代码运行正确，最后一行应输出 `验证结果： True`。这说明验证者在不知道明文 `x` 的情况下，确认了 Alice 的证明满足公开命题 `x^3 + x + 5 = 35`。

In [8]:
# 篡改测试 1：把公开 out 从 35 改成 36。
tampered_out_proof = dict(proof)
tampered_out_proof["out"] = 36
print("篡改 out 后的验证结果：", verify_polynomial_proof(tampered_out_proof))

# 篡改测试 2：改动证明中的一个响应 z。
tampered_z_proof = dict(proof)
tampered_z_proof["cp1"] = dict(proof["cp1"])
tampered_z_proof["cp1"]["z"] = (tampered_z_proof["cp1"]["z"] + 1) % N
print("篡改 z 后的验证结果：", verify_polynomial_proof(tampered_z_proof))

# 错误见证测试：x = 2 不满足 2^3 + 2 + 5 = 35，诚实证明生成函数会拒绝。
try:
    wrong_proof = generate_polynomial_proof(2, 35)
except ValueError as error:
    print("错误见证生成证明失败：", error)

篡改 out 后的验证结果： False
篡改 z 后的验证结果： False
错误见证生成证明失败： 见证 x 不满足 x^3 + x + 5 = out，不能生成诚实证明。


## 2. 实验流程总结

1. 初始化公开参数：选择椭圆曲线 secp256k1 和公开基点 `G`。
2. Alice 准备秘密见证：令 `x = 3`，公开值为 `out = 35`。
3. Alice 计算曲线点：`A = xG`，`B = xA`，`C = xB`。
4. Alice 生成两个非交互式 Chaum-Pedersen 证明，证明 `A、B、C` 都由同一个 `x` 关联生成。
5. 验证者验证两个子证明，并检查 `C + A + 5G == outG`。
6. 对证明进行篡改测试，观察验证是否失败。

## 3. 如何分析实验结果是否正确

- 正确证明应输出 `验证结果： True`。这对应完备性：Alice 真的知道满足方程的 `x = 3`，证明应通过。
- 把 `out` 改成 `36` 后应输出 `False`。因为 `3^3 + 3 + 5` 不等于 `36`，曲线上的等式 `C + A + 5G == 36G` 不成立。
- 改动证明响应 `z` 后应输出 `False`。因为 Fiat-Shamir 挑战和响应被破坏，Schnorr 型验证等式不再成立。
- 用 `x = 2` 生成证明会直接失败。因为 `2^3 + 2 + 5 = 15`，不满足公开声明 `out = 35`。

因此，若正确样例通过、篡改样例失败、错误见证被拒绝，就说明本实验的证明生成和验证流程符合预期。

## 4. 安全性说明

本实验实现的是教学版协议，用于理解教材中的 Schnorr、Fiat-Shamir 和零知识证明流程。它不是完整 zkSNARK，也没有经过工程级安全审计。实际业务系统中应使用成熟密码库，并选择更严格的电路约束、序列化格式、随机数生成和安全参数。